In [1]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn joblib
%pip install -q transformers datasets accelerate torch

In [2]:
import os
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Set these manually if auto-detection fails
DATA_PATH = None          # Example: "data/train.csv"
TEXT_COL = None           # Example: "text"
LABEL_COL = None          # Example: "label"

# Your fixed class order
CLASS_NAMES = ['grammar', 'structure', 'clarity', 'evidence', 'argument', 'other', 'style']

# Auto-search patterns if DATA_PATH is None
SEARCH_PATTERNS = ["**/*.csv", "**/*.parquet"]

print("Config loaded.")

Config loaded.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

root = Path("/content/drive/MyDrive")
for p in root.rglob("*.zip"):
    name = p.name.lower()
    if "feedback-prize" in name or "patent" in name:
        print(p)

/content/drive/MyDrive/us-patent-phrase-to-phrase-matching.zip
/content/drive/MyDrive/feedback-prize-2021.zip
/content/drive/MyDrive/feedback-prize-effectiveness.zip
/content/drive/MyDrive/feedback-prize-english-language-learning.zip


In [5]:
import zipfile
from pathlib import Path

ZIP_FILES = {
    "fp2021": "/content/drive/MyDrive/feedback-prize-2021.zip",
    "fpe": "/content/drive/MyDrive/feedback-prize-effectiveness.zip",
    "ell": "/content/drive/MyDrive/feedback-prize-english-language-learning.zip",
    "patent": "/content/drive/MyDrive/us-patent-phrase-to-phrase-matching.zip",
}

EXTRACT_ROOT = Path("/content/eduai_datasets")
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

for name, zip_path in ZIP_FILES.items():
    zip_path = Path(zip_path)
    if not zip_path.exists():
        print(f"Missing: {zip_path}")
        continue

    out_dir = EXTRACT_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    print(f"Extracted {name} -> {out_dir}")

Extracted fp2021 -> /content/eduai_datasets/fp2021
Extracted fpe -> /content/eduai_datasets/fpe
Extracted ell -> /content/eduai_datasets/ell
Extracted patent -> /content/eduai_datasets/patent


In [6]:
from pathlib import Path
import pandas as pd

EXTRACT_ROOT = Path("/content/eduai_datasets")

def list_files(root, max_files=50):
    files = [p for p in root.rglob("*") if p.is_file()]
    print(f"\n{root} -> {len(files)} files")
    for p in files[:max_files]:
        print(" -", p)

for name in ["fp2021", "fpe", "ell", "patent"]:
    folder = EXTRACT_ROOT / name
    list_files(folder, max_files=30)


/content/eduai_datasets/fp2021 -> 15601 files
 - /content/eduai_datasets/fp2021/train.csv
 - /content/eduai_datasets/fp2021/sample_submission.csv
 - /content/eduai_datasets/fp2021/test/DF920E0A7337.txt
 - /content/eduai_datasets/fp2021/test/D72CB1C11673.txt
 - /content/eduai_datasets/fp2021/test/18409261F5C2.txt
 - /content/eduai_datasets/fp2021/test/0FB0700DAF44.txt
 - /content/eduai_datasets/fp2021/test/D46BCB48440A.txt
 - /content/eduai_datasets/fp2021/train/423A1CA112E2.txt
 - /content/eduai_datasets/fp2021/train/C1043F3C0609.txt
 - /content/eduai_datasets/fp2021/train/AF34B1CDD152.txt
 - /content/eduai_datasets/fp2021/train/E5C8A407F6EE.txt
 - /content/eduai_datasets/fp2021/train/E383A10B9A09.txt
 - /content/eduai_datasets/fp2021/train/EA7403CC3053.txt
 - /content/eduai_datasets/fp2021/train/53A912957FE2.txt
 - /content/eduai_datasets/fp2021/train/4EF6C70DC809.txt
 - /content/eduai_datasets/fp2021/train/B67D550009E8.txt
 - /content/eduai_datasets/fp2021/train/DF9C8AF077C6.txt
 - 

In [7]:
from pathlib import Path

EXTRACT_ROOT = Path("/content/eduai_datasets")

def find_first(root: Path, pattern: str):
    matches = list(root.rglob(pattern))
    if not matches:
        return None
    return matches[0]

fp2021_root = EXTRACT_ROOT / "fp2021"
fpe_root = EXTRACT_ROOT / "fpe"
ell_root = EXTRACT_ROOT / "ell"

fp2021_train_csv = find_first(fp2021_root, "train.csv")
fpe_train_csv = find_first(fpe_root, "train.csv")
ell_train_csv = find_first(ell_root, "train.csv")

# For FP2021, the essay text directory is usually named "train"
fp2021_essay_dir = None
for p in fp2021_root.rglob("*"):
    if p.is_dir() and p.name.lower() == "train":
        # make sure it contains txt files
        txts = list(p.glob("*.txt"))
        if txts:
            fp2021_essay_dir = p
            break

print("fp2021_train_csv :", fp2021_train_csv)
print("fp2021_essay_dir :", fp2021_essay_dir)
print("fpe_train_csv    :", fpe_train_csv)
print("ell_train_csv    :", ell_train_csv)

fp2021_train_csv : /content/eduai_datasets/fp2021/train.csv
fp2021_essay_dir : /content/eduai_datasets/fp2021/train
fpe_train_csv    : /content/eduai_datasets/fpe/train.csv
ell_train_csv    : /content/eduai_datasets/ell/train.csv


In [8]:
from pathlib import Path
import pandas as pd

EXTRACT_ROOT = Path("/content/eduai_datasets")

def find_first(root: Path, pattern: str):
    matches = list(root.rglob(pattern))
    return matches[0] if matches else None

fp2021_root = EXTRACT_ROOT / "fp2021"
fpe_root = EXTRACT_ROOT / "fpe"
ell_root = EXTRACT_ROOT / "ell"

fp2021_train_csv = find_first(fp2021_root, "train.csv")
fpe_train_csv = find_first(fpe_root, "train.csv")
ell_train_csv = find_first(ell_root, "train.csv")

print("fp2021_train_csv:", fp2021_train_csv)
print("fpe_train_csv   :", fpe_train_csv)
print("ell_train_csv   :", ell_train_csv)

fp2021_train = pd.read_csv(fp2021_train_csv)
fpe_train = pd.read_csv(fpe_train_csv)
ell_train = pd.read_csv(ell_train_csv)

print("FP2021 shape:", fp2021_train.shape)
print("FPE shape   :", fpe_train.shape)
print("ELL shape   :", ell_train.shape)

fp2021_train_csv: /content/eduai_datasets/fp2021/train.csv
fpe_train_csv   : /content/eduai_datasets/fpe/train.csv
ell_train_csv   : /content/eduai_datasets/ell/train.csv
FP2021 shape: (144293, 8)
FPE shape   : (36765, 5)
ELL shape   : (3911, 8)


In [9]:
print("FP2021 discourse types:")
print(fp2021_train["discourse_type"].value_counts(dropna=False))

print("\nFPE discourse types:")
print(fpe_train["discourse_type"].value_counts(dropna=False))

print("\nFPE effectiveness:")
print(fpe_train["discourse_effectiveness"].value_counts(dropna=False))

FP2021 discourse types:


discourse_type
Claim                   50208
Evidence                45702
Position                15419
Concluding Statement    13505
Lead                     9305
Counterclaim             5817
Rebuttal                 4337
Name: count, dtype: int64

FPE discourse types:
discourse_type
Evidence                12105
Claim                   11977
Position                 4024
Concluding Statement     3351
Lead                     2291
Counterclaim             1773
Rebuttal                 1244
Name: count, dtype: int64

FPE effectiveness:
discourse_effectiveness
Adequate       20977
Effective       9326
Ineffective     6462
Name: count, dtype: int64


In [10]:
from pathlib import Path

EXTRACT_ROOT = Path("/content/eduai_datasets")

fp2021_essay_dir = None
for p in (EXTRACT_ROOT / "fp2021").rglob("*"):
    if p.is_dir() and p.name.lower() == "train":
        txts = list(p.glob("*.txt"))
        if txts:
            fp2021_essay_dir = p
            break

print("FP2021 essay dir:", fp2021_essay_dir)
print("Number of txt essays:", len(list(fp2021_essay_dir.glob("*.txt"))))

FP2021 essay dir: /content/eduai_datasets/fp2021/train
Number of txt essays: 15594


In [11]:
import re
import random
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = ['grammar', 'structure', 'clarity', 'evidence', 'argument', 'other', 'style']

def clean_text(text: str) -> str:
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text: str) -> int:
    return len(clean_text(text).split())

def clip_text(text: str, max_chars: int = 1800) -> str:
    return clean_text(text)[:max_chars]

def keep_text(text: str, min_words: int = 6, max_words: int = 400) -> bool:
    n = word_count(text)
    return min_words <= n <= max_words

In [12]:
FP_MAP = {
    "Evidence": "evidence",
    "Claim": "argument",
    "Counterclaim": "argument",
    "Rebuttal": "argument",
    "Position": "argument",
    "Lead": "structure",
    "Concluding Statement": "structure",
}

fp2021_mapped = fp2021_train.copy()
fp2021_mapped["label"] = fp2021_mapped["discourse_type"].map(FP_MAP)
fp2021_mapped = fp2021_mapped[fp2021_mapped["label"].notna()].copy()

fp2021_samples = pd.DataFrame({
    "text": fp2021_mapped["discourse_text"].map(clean_text),
    "label": fp2021_mapped["label"],
    "source": "fp2021_discourse",
    "weight": 1.0
})

fp2021_samples = fp2021_samples[fp2021_samples["text"].map(keep_text)].copy()

print("FP2021 mapped samples:", fp2021_samples.shape)
print(fp2021_samples["label"].value_counts())
display(fp2021_samples.head())

FP2021 mapped samples: (139190, 4)
label
argument     70805
evidence     45625
structure    22760
Name: count, dtype: int64


,text,label,source,weight
0,Modern humans today are always on their phone....,structure,fp2021_discourse,1.0
1,They are some really bad consequences when stu...,argument,fp2021_discourse,1.0
2,Some certain areas in the United States ban ph...,evidence,fp2021_discourse,1.0
3,"When people have phones, they know about certa...",evidence,fp2021_discourse,1.0
4,Driving is one of the way how to get around. P...,argument,fp2021_discourse,1.0


In [13]:
essay_texts = {}
for txt_file in fp2021_essay_dir.glob("*.txt"):
    essay_id = txt_file.stem
    essay_texts[essay_id] = txt_file.read_text(encoding="utf-8", errors="ignore")

print("Loaded essay texts:", len(essay_texts))

Loaded essay texts: 15594


In [14]:
def extract_other_segments_from_essay(essay_text, spans, min_words=6, max_words=120):
    segments = []
    prev_end = 0

    for start, end in spans:
        if start > prev_end:
            gap = essay_text[prev_end:start]
            gap = clean_text(gap)
            wc = word_count(gap)
            if min_words <= wc <= max_words:
                segments.append(gap)
        prev_end = max(prev_end, end)

    if prev_end < len(essay_text):
        gap = essay_text[prev_end:]
        gap = clean_text(gap)
        wc = word_count(gap)
        if min_words <= wc <= max_words:
            segments.append(gap)

    return segments

In [15]:
other_rows = []

for essay_id, group in fp2021_train.groupby("id"):
    essay_text = essay_texts.get(str(essay_id))
    if not essay_text:
        continue

    spans = (
        group[["discourse_start", "discourse_end"]]
        .dropna()
        .sort_values("discourse_start")
        .astype(int)
        .values
        .tolist()
    )

    gaps = extract_other_segments_from_essay(essay_text, spans)

    for g in gaps:
        other_rows.append({
            "text": g,
            "label": "other",
            "source": "fp2021_gap_other",
            "weight": 0.9
        })

other_df = pd.DataFrame(other_rows).drop_duplicates(subset=["text"]).copy()

print("Other samples:", other_df.shape)
display(other_df.head())

Other samples: (11194, 4)


,text,label,source,weight
0,One thing that the article mentions is that,other,fp2021_gap_other,0.9
1,One last thing that the article mentions is ne...,other,fp2021_gap_other,0.9
2,Some reasons I would be against the school pol...,other,fp2021_gap_other,0.9
3,. taking only classes helps them because at th...,other,fp2021_gap_other,0.9
4,. May there be many ways that,other,fp2021_gap_other,0.9


In [16]:
EFFECTIVENESS_WEIGHT = {
    "Effective": 1.20,
    "Adequate": 1.00,
    "Ineffective": 0.75,
}

fpe_mapped = fpe_train.copy()
fpe_mapped["label"] = fpe_mapped["discourse_type"].map(FP_MAP)
fpe_mapped = fpe_mapped[fpe_mapped["label"].notna()].copy()

fpe_samples = pd.DataFrame({
    "text": fpe_mapped["discourse_text"].map(clean_text),
    "label": fpe_mapped["label"],
    "source": "fpe_discourse",
    "weight": fpe_mapped["discourse_effectiveness"].map(EFFECTIVENESS_WEIGHT).fillna(1.0)
})

fpe_samples = fpe_samples[fpe_samples["text"].map(keep_text)].copy()

print("FPE mapped samples:", fpe_samples.shape)
print(fpe_samples["label"].value_counts())
display(fpe_samples.head())

FPE mapped samples: (35711, 4)
label
argument     18021
evidence     12068
structure     5622
Name: count, dtype: int64


,text,label,source,weight
0,"Hi, i'm Isaac, i'm going to be writing about h...",structure,fpe_discourse,1.0
1,"On my perspective, I think that the face is a ...",argument,fpe_discourse,1.0
2,I think that the face is a natural landform be...,argument,fpe_discourse,1.0
3,"If life was on Mars, we would know by now. The...",evidence,fpe_discourse,1.0
4,People thought that the face was formed by ali...,argument,fpe_discourse,1.0


In [17]:
ell = ell_train.copy()

ell["grammar_signal"] = (ell["grammar"] + ell["conventions"]) / 2.0
ell["structure_signal"] = ell["cohesion"]
ell["clarity_signal"] = ell["syntax"]
ell["style_signal"] = (ell["vocabulary"] + ell["phraseology"]) / 2.0

signal_cols = ["grammar_signal", "structure_signal", "clarity_signal", "style_signal"]
signal_to_label = {
    "grammar_signal": "grammar",
    "structure_signal": "structure",
    "clarity_signal": "clarity",
    "style_signal": "style",
}

def assign_weak_ell_label(row, low_threshold=2.5, margin=0.35):
    vals = {c: float(row[c]) for c in signal_cols}
    sorted_items = sorted(vals.items(), key=lambda x: x[1])

    best_name, best_val = sorted_items[0]
    second_name, second_val = sorted_items[1]

    if best_val <= low_threshold and (second_val - best_val) >= margin:
        return signal_to_label[best_name]
    return None

ell["label"] = ell.apply(assign_weak_ell_label, axis=1)

ell_samples = ell[ell["label"].notna()].copy()
ell_samples = pd.DataFrame({
    "text": ell_samples["full_text"].map(lambda x: clip_text(x, 1800)),
    "label": ell_samples["label"],
    "source": "ell_weak",
    "weight": 0.55
})

ell_samples = ell_samples[ell_samples["text"].map(keep_text)].copy()

print("ELL weak samples:", ell_samples.shape)
print(ell_samples["label"].value_counts())
display(ell_samples.head())

ELL weak samples: (603, 4)
label
clarity      236
structure    207
grammar      135
style         25
Name: count, dtype: int64


,text,label,source,weight
12,Technology allows people to do many things suc...,clarity,ell_weak,0.55
16,A positive attitude is the key for be successf...,clarity,ell_weak,0.55
18,"March 12, 20019 The technology allows people t...",clarity,ell_weak,0.55
34,I would agree with being honest at all times b...,structure,ell_weak,0.55
36,Do you think positive attitude is the key to s...,clarity,ell_weak,0.55


In [18]:
train_df = pd.concat(
    [fp2021_samples, other_df, fpe_samples, ell_samples],
    ignore_index=True
)

train_df["text"] = train_df["text"].map(clean_text)
train_df = train_df[train_df["label"].isin(CLASS_NAMES)].copy()
train_df = train_df[train_df["text"].str.len() > 0].copy()
train_df = train_df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("Combined dataset shape:", train_df.shape)
print("\nLabel distribution:")
print(train_df["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))

print("\nSource distribution:")
print(train_df["source"].value_counts())

display(train_df.sample(min(10, len(train_df)), random_state=SEED))

Combined dataset shape: (150502, 4)

Label distribution:
label
grammar        135
structure    22924
clarity        236
evidence     45607
argument     70381
other        11194
style           25
Name: count, dtype: int64

Source distribution:
source
fp2021_discourse    138705
fp2021_gap_other     11194
ell_weak               603
Name: count, dtype: int64


,text,label,source,weight
70263,If we had a D or an F i would understand that,argument,fp2021_discourse,1.0
69478,cause them to be less healthy,argument,fp2021_discourse,1.0
77951,In paragraph nine the law is the issue that is...,argument,fp2021_discourse,1.0
32275,When having to put another thing on top of sch...,evidence,fp2021_discourse,1.0
138704,there are many other reasons one might want to...,structure,fp2021_discourse,1.0
23168,"In young adult life, such as the high school y...",structure,fp2021_discourse,1.0
88532,I strongly agree of keeping the Electoral Coll...,argument,fp2021_discourse,1.0
100595,Cell phones are huge distractions in school. T...,structure,fp2021_discourse,1.0
83242,they could create a lot of oppritunities for s...,argument,fp2021_discourse,1.0
134916,Some people actually think it helps telling a ...,argument,fp2021_discourse,1.0


In [19]:
ell = ell_train.copy()

ell["grammar_signal"] = (ell["grammar"] + ell["conventions"]) / 2.0
ell["clarity_signal"] = ell["syntax"]
ell["style_signal"] = (ell["vocabulary"] + ell["phraseology"]) / 2.0

LANG_SIGNAL_COLS = ["grammar_signal", "clarity_signal", "style_signal"]
LANG_SIGNAL_TO_LABEL = {
    "grammar_signal": "grammar",
    "clarity_signal": "clarity",
    "style_signal": "style",
}

def assign_language_label_v2(row):
    vals = {c: float(row[c]) for c in LANG_SIGNAL_COLS}
    weakest_signal = min(vals, key=vals.get)   # lowest score = weakest area
    return LANG_SIGNAL_TO_LABEL[weakest_signal]

ell["label"] = ell.apply(assign_language_label_v2, axis=1)

ell_samples_v2 = pd.DataFrame({
    "text": ell["full_text"].map(lambda x: clip_text(x, 1800)),
    "label": ell["label"],
    "source": "ell_language_v2",
    "weight": 0.40
})

ell_samples_v2 = ell_samples_v2[ell_samples_v2["text"].map(keep_text)].copy()

print("ELL v2 samples:", ell_samples_v2.shape)
print(ell_samples_v2["label"].value_counts())
display(ell_samples_v2.head())

ELL v2 samples: (3909, 4)
label
grammar    1939
clarity    1485
style       485
Name: count, dtype: int64


,text,label,source,weight
0,I think that students would benefit from learn...,style,ell_language_v2,0.4
1,When a problem is a change you have to let it ...,grammar,ell_language_v2,0.4
2,"Dear, Principal If u change the school policy ...",grammar,ell_language_v2,0.4
3,The best time in life is when you become yours...,grammar,ell_language_v2,0.4
4,Small act of kindness can impact in other peop...,grammar,ell_language_v2,0.4


In [20]:
combined_df = pd.concat(
    [fp2021_samples, other_df, fpe_samples, ell_samples_v2],
    ignore_index=True
)

combined_df["text"] = combined_df["text"].map(clean_text)
combined_df = combined_df[combined_df["label"].isin(CLASS_NAMES)].copy()
combined_df = combined_df[combined_df["text"].str.len() > 0].copy()

# Aggregate duplicates instead of dropping them blindly
combined_df = (
    combined_df
    .groupby(["text", "label"], as_index=False)
    .agg(
        weight=("weight", "mean"),
        source=("source", lambda s: "|".join(sorted(set(s))))
    )
)

print("Combined dataset shape:", combined_df.shape)
print("\nLabel distribution:")
print(combined_df["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))

print("\nExample sources:")
display(combined_df["source"].value_counts().head(10))
display(combined_df.sample(min(10, len(combined_df)), random_state=SEED))

Combined dataset shape: (153808, 4)

Label distribution:
label
grammar       1939
structure    22717
clarity       1485
evidence     45607
argument     70381
other        11194
style          485
Name: count, dtype: int64

Example sources:


,count
source,
fp2021_discourse,103032
fp2021_discourse|fpe_discourse,35673
fp2021_gap_other,11194
ell_language_v2,3909


,text,label,weight,source
86094,Some students would not benefit learning from ...,argument,1.0,fp2021_discourse
71849,Now you're six feet under just because you wan...,evidence,1.0,fp2021_discourse
143343,probably more students would get better grades...,argument,1.0,fp2021_discourse|fpe_discourse
28542,For example when I asked my mom on advice abou...,evidence,1.0,fp2021_discourse
133925,different people give you good advice it will ...,evidence,1.0,fp2021_discourse
33998,Homeschooling is a very benificial way to get ...,structure,1.0,fp2021_discourse|fpe_discourse
14035,Asking more then one person will make you feel...,argument,1.0,fp2021_discourse
1472,", pollution from cars has been a large factor ...",argument,1.0,fp2021_discourse
142120,no coach would want a knucklehead on there team.,argument,1.0,fp2021_discourse
149543,"they have less greenhouse gas emissions,",argument,1.0,fp2021_discourse|fpe_discourse


In [21]:
from sklearn.model_selection import train_test_split

train_part, temp_part = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df["label"],
    random_state=SEED
)

val_part, test_part = train_test_split(
    temp_part,
    test_size=0.50,
    stratify=temp_part["label"],
    random_state=SEED
)

print("Before train rebalance")
print("Train:", train_part.shape)
print(train_part["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))
print("\nVal:", val_part.shape)
print(val_part["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))
print("\nTest:", test_part.shape)
print(test_part["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))

Before train rebalance
Train: (123046, 4)
label
grammar       1551
structure    18174
clarity       1188
evidence     36485
argument     56305
other         8955
style          388
Name: count, dtype: int64

Val: (15381, 4)
label
grammar       194
structure    2271
clarity       148
evidence     4561
argument     7038
other        1120
style          49
Name: count, dtype: int64

Test: (15381, 4)
label
grammar       194
structure    2272
clarity       149
evidence     4561
argument     7038
other        1119
style          48
Name: count, dtype: int64


In [22]:
def rebalance_train(df, min_per_class=1800, max_per_class=14000, seed=42):
    parts = []

    for label, group in df.groupby("label"):
        if len(group) > max_per_class:
            group = group.sample(max_per_class, random_state=seed)
        elif len(group) < min_per_class:
            group = group.sample(min_per_class, replace=True, random_state=seed)

        parts.append(group)

    out = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

train_bal = rebalance_train(train_part, min_per_class=1800, max_per_class=14000, seed=SEED)

print("After train rebalance")
print(train_bal.shape)
print(train_bal["label"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))

After train rebalance
(56355, 4)
label
grammar       1800
structure    14000
clarity       1800
evidence     14000
argument     14000
other         8955
style         1800
Name: count, dtype: int64


In [23]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_model(name, model, X, y):
    preds = model.predict(X)

    acc = accuracy_score(y, preds)
    macro_f1 = f1_score(y, preds, average="macro")
    weighted_f1 = f1_score(y, preds, average="weighted")

    print(f"\n{'='*80}")
    print(name)
    print(f"{'='*80}")
    print(f"Accuracy   : {acc:.4f}")
    print(f"Macro F1   : {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print("\nClassification report:\n")
    print(classification_report(y, preds, labels=CLASS_NAMES, digits=4))

    return preds, macro_f1

feature_block = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="word",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=120000
    )),
    ("char_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=3,
        sublinear_tf=True,
        max_features=80000
    ))
])

In [24]:
svm_pipeline = Pipeline([
    ("features", feature_block),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=SEED
    ))
])

svm_pipeline.fit(
    train_bal["text"],
    train_bal["label"],
    clf__sample_weight=train_bal["weight"].values
)

svm_preds, svm_macro = evaluate_model(
    "Word+Char TF-IDF + LinearSVC",
    svm_pipeline,
    val_part["text"],
    val_part["label"]
)


Word+Char TF-IDF + LinearSVC
Accuracy   : 0.6683
Macro F1   : 0.4886
Weighted F1: 0.6786

Classification report:

              precision    recall  f1-score   support

     grammar     0.4946    0.4691    0.4815       194
   structure     0.5256    0.7094    0.6038      2271
     clarity     0.3561    0.3176    0.3357       148
    evidence     0.7368    0.7034    0.7197      4561
    argument     0.8164    0.6654    0.7332      7038
       other     0.3376    0.5661    0.4229      1120
       style     0.1562    0.1020    0.1235        49

    accuracy                         0.6683     15381
   macro avg     0.4890    0.5047    0.4886     15381
weighted avg     0.7044    0.6683    0.6786     15381



In [25]:
discourse_labels = ["structure", "evidence", "argument", "other"]

disc_df = combined_df[combined_df["label"].isin(discourse_labels)].copy()

print("Discourse dataset shape:", disc_df.shape)
print(disc_df["label"].value_counts())

Discourse dataset shape: (149899, 4)
label
argument     70381
evidence     45607
structure    22717
other        11194
Name: count, dtype: int64


In [26]:
from sklearn.model_selection import train_test_split

disc_train, disc_temp = train_test_split(
    disc_df,
    test_size=0.20,
    stratify=disc_df["label"],
    random_state=SEED
)

disc_val, disc_test = train_test_split(
    disc_temp,
    test_size=0.50,
    stratify=disc_temp["label"],
    random_state=SEED
)

print("Train:", disc_train.shape)
print(disc_train["label"].value_counts())
print("\nVal:", disc_val.shape)
print(disc_val["label"].value_counts())
print("\nTest:", disc_test.shape)
print(disc_test["label"].value_counts())

Train: (119919, 4)
label
argument     56305
evidence     36485
structure    18174
other         8955
Name: count, dtype: int64

Val: (14990, 4)
label
argument     7038
evidence     4561
structure    2272
other        1119
Name: count, dtype: int64

Test: (14990, 4)
label
argument     7038
evidence     4561
structure    2271
other        1120
Name: count, dtype: int64


In [27]:
def rebalance_train(df, min_per_class=5000, max_per_class=16000, seed=42):
    parts = []
    for label, group in df.groupby("label"):
        if len(group) > max_per_class:
            group = group.sample(max_per_class, random_state=seed)
        elif len(group) < min_per_class:
            group = group.sample(min_per_class, replace=True, random_state=seed)
        parts.append(group)
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

disc_train_bal = rebalance_train(disc_train, min_per_class=5000, max_per_class=16000, seed=SEED)

print(disc_train_bal.shape)
print(disc_train_bal["label"].value_counts())

(56955, 4)
label
argument     16000
structure    16000
evidence     16000
other         8955
Name: count, dtype: int64


In [28]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

disc_feature_block = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="word",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=120000
    )),
    ("char_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=3,
        sublinear_tf=True,
        max_features=60000
    ))
])

disc_model = Pipeline([
    ("features", disc_feature_block),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=SEED
    ))
])

disc_model.fit(
    disc_train_bal["text"],
    disc_train_bal["label"],
    clf__sample_weight=disc_train_bal["weight"].values
)

disc_val_preds = disc_model.predict(disc_val["text"])

print("Discourse Validation Accuracy   :", accuracy_score(disc_val["label"], disc_val_preds))
print("Discourse Validation Macro F1   :", f1_score(disc_val["label"], disc_val_preds, average="macro"))
print("Discourse Validation Weighted F1:", f1_score(disc_val["label"], disc_val_preds, average="weighted"))
print("\nValidation report:\n")
print(classification_report(disc_val["label"], disc_val_preds, labels=discourse_labels, digits=4))

Discourse Validation Accuracy   : 0.6785857238158772
Discourse Validation Macro F1   : 0.6183820576587
Discourse Validation Weighted F1: 0.6885885775574804

Validation report:

              precision    recall  f1-score   support

   structure     0.5090    0.6835    0.5835      2272
    evidence     0.7418    0.7174    0.7294      4561
    argument     0.8143    0.6755    0.7384      7038
       other     0.3509    0.5299    0.4222      1119

    accuracy                         0.6786     14990
   macro avg     0.6040    0.6516    0.6184     14990
weighted avg     0.7114    0.6786    0.6886     14990



In [29]:
disc_test_preds = disc_model.predict(disc_test["text"])

print("Discourse Test Accuracy   :", accuracy_score(disc_test["label"], disc_test_preds))
print("Discourse Test Macro F1   :", f1_score(disc_test["label"], disc_test_preds, average="macro"))
print("Discourse Test Weighted F1:", f1_score(disc_test["label"], disc_test_preds, average="weighted"))
print("\nTest report:\n")
print(classification_report(disc_test["label"], disc_test_preds, labels=discourse_labels, digits=4))

Discourse Test Accuracy   : 0.6863909272848566
Discourse Test Macro F1   : 0.6274684421995353
Discourse Test Weighted F1: 0.6965589111867715

Test report:

              precision    recall  f1-score   support

   structure     0.5325    0.6997    0.6048      2271
    evidence     0.7531    0.7268    0.7397      4561
    argument     0.8184    0.6780    0.7416      7038
       other     0.3457    0.5473    0.4238      1120

    accuracy                         0.6864     14990
   macro avg     0.6124    0.6630    0.6275     14990
weighted avg     0.7199    0.6864    0.6966     14990



In [30]:
lang_labels = ["grammar", "clarity", "style"]

lang_df = ell_samples_v2.copy()
lang_df = lang_df[lang_df["label"].isin(lang_labels)].copy()

print("Language dataset shape:", lang_df.shape)
print(lang_df["label"].value_counts())

Language dataset shape: (3909, 4)
label
grammar    1939
clarity    1485
style       485
Name: count, dtype: int64


In [31]:
lang_train, lang_temp = train_test_split(
    lang_df,
    test_size=0.20,
    stratify=lang_df["label"],
    random_state=SEED
)

lang_val, lang_test = train_test_split(
    lang_temp,
    test_size=0.50,
    stratify=lang_temp["label"],
    random_state=SEED
)

print("Train:", lang_train.shape)
print(lang_train["label"].value_counts())
print("\nVal:", lang_val.shape)
print(lang_val["label"].value_counts())
print("\nTest:", lang_test.shape)
print(lang_test["label"].value_counts())

Train: (3127, 4)
label
grammar    1551
clarity    1188
style       388
Name: count, dtype: int64

Val: (391, 4)
label
grammar    194
clarity    148
style       49
Name: count, dtype: int64

Test: (391, 4)
label
grammar    194
clarity    149
style       48
Name: count, dtype: int64


In [32]:
lang_train_bal = rebalance_train(lang_train, min_per_class=1200, max_per_class=2500, seed=SEED)

print(lang_train_bal.shape)
print(lang_train_bal["label"].value_counts())

(3951, 4)
label
grammar    1551
style      1200
clarity    1200
Name: count, dtype: int64


In [33]:
lang_feature_block = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=50000
    )),
    ("char_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=60000
    ))
])

lang_model = Pipeline([
    ("features", lang_feature_block),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=SEED
    ))
])

lang_model.fit(
    lang_train_bal["text"],
    lang_train_bal["label"],
    clf__sample_weight=lang_train_bal["weight"].values
)

lang_val_preds = lang_model.predict(lang_val["text"])

print("Language Validation Accuracy   :", accuracy_score(lang_val["label"], lang_val_preds))
print("Language Validation Macro F1   :", f1_score(lang_val["label"], lang_val_preds, average="macro"))
print("Language Validation Weighted F1:", f1_score(lang_val["label"], lang_val_preds, average="weighted"))
print("\nValidation report:\n")
print(classification_report(lang_val["label"], lang_val_preds, labels=lang_labels, digits=4))

Language Validation Accuracy   : 0.47058823529411764
Language Validation Macro F1   : 0.322439372944353
Language Validation Weighted F1: 0.41788678598069984

Validation report:

              precision    recall  f1-score   support

     grammar     0.5118    0.7835    0.6191       194
     clarity     0.4085    0.1959    0.2648       148
       style     0.1304    0.0612    0.0833        49

    accuracy                         0.4706       391
   macro avg     0.3502    0.3469    0.3224       391
weighted avg     0.4249    0.4706    0.4179       391



In [34]:
ell_reg = ell_train.copy()

ell_reg["text"] = ell_reg["full_text"].map(lambda x: clip_text(x, 1800))
ell_reg = ell_reg[ell_reg["text"].map(keep_text)].copy()

ell_reg["grammar_target"] = (ell_reg["grammar"] + ell_reg["conventions"]) / 2.0
ell_reg["clarity_target"] = ell_reg["syntax"]
ell_reg["style_target"] = (ell_reg["vocabulary"] + ell_reg["phraseology"]) / 2.0

print(ell_reg.shape)
display(
    ell_reg[["grammar_target", "clarity_target", "style_target"]].describe()
)
display(ell_reg[["text", "grammar_target", "clarity_target", "style_target"]].head())

(3909, 12)


,grammar_target,clarity_target,style_target
count,3909.000000,3909.000000,3909.000000
mean,3.057304,3.028652,3.176644
std,0.627141,0.644298,0.577098
min,1.000000,1.000000,1.000000
25%,2.500000,2.500000,2.750000
50%,3.000000,3.000000,3.250000
75%,3.500000,3.500000,3.500000
max,5.000000,5.000000,5.000000


,text,grammar_target,clarity_target,style_target
0,I think that students would benefit from learn...,3.50,3.5,3.0
1,When a problem is a change you have to let it ...,2.25,2.5,2.5
2,"Dear, Principal If u change the school policy ...",2.75,3.5,3.0
3,The best time in life is when you become yours...,4.50,4.5,4.5
4,Small act of kindness can impact in other peop...,2.50,3.0,3.0


In [35]:
from sklearn.model_selection import train_test_split

ell_train_part, ell_temp_part = train_test_split(
    ell_reg,
    test_size=0.20,
    random_state=SEED
)

ell_val_part, ell_test_part = train_test_split(
    ell_temp_part,
    test_size=0.50,
    random_state=SEED
)

print("Train:", ell_train_part.shape)
print("Val  :", ell_val_part.shape)
print("Test :", ell_test_part.shape)

Train: (3127, 12)
Val  : (391, 12)
Test : (391, 12)


In [36]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

reg_feature_block = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=60000
    )),
    ("char_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=60000
    ))
])

def evaluate_regression(name, model, X, y_true):
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)

    print(f"\n{'='*80}")
    print(name)
    print(f"{'='*80}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R²  : {r2:.4f}")
    return preds, rmse, mae, r2

In [37]:
grammar_reg = Pipeline([
    ("features", reg_feature_block),
    ("reg", Ridge(alpha=3.0, random_state=SEED))
])

grammar_reg.fit(
    ell_train_part["text"],
    ell_train_part["grammar_target"]
)

grammar_val_preds, grammar_rmse, grammar_mae, grammar_r2 = evaluate_regression(
    "Grammar Regressor",
    grammar_reg,
    ell_val_part["text"],
    ell_val_part["grammar_target"]
)


Grammar Regressor
RMSE: 0.4963
MAE : 0.3939
R²  : 0.3611


In [38]:
clarity_reg = Pipeline([
    ("features", reg_feature_block),
    ("reg", Ridge(alpha=3.0, random_state=SEED))
])

clarity_reg.fit(
    ell_train_part["text"],
    ell_train_part["clarity_target"]
)

clarity_val_preds, clarity_rmse, clarity_mae, clarity_r2 = evaluate_regression(
    "Clarity Regressor",
    clarity_reg,
    ell_val_part["text"],
    ell_val_part["clarity_target"]
)


Clarity Regressor
RMSE: 0.5210
MAE : 0.4233
R²  : 0.3381


In [39]:
style_reg = Pipeline([
    ("features", reg_feature_block),
    ("reg", Ridge(alpha=3.0, random_state=SEED))
])

style_reg.fit(
    ell_train_part["text"],
    ell_train_part["style_target"]
)

style_val_preds, style_rmse, style_mae, style_r2 = evaluate_regression(
    "Style Regressor",
    style_reg,
    ell_val_part["text"],
    ell_val_part["style_target"]
)


Style Regressor
RMSE: 0.4384
MAE : 0.3423
R²  : 0.3833


In [40]:
print("Grammar test:")
_ = evaluate_regression(
    "Grammar Regressor - Test",
    grammar_reg,
    ell_test_part["text"],
    ell_test_part["grammar_target"]
)

print("\nClarity test:")
_ = evaluate_regression(
    "Clarity Regressor - Test",
    clarity_reg,
    ell_test_part["text"],
    ell_test_part["clarity_target"]
)

print("\nStyle test:")
_ = evaluate_regression(
    "Style Regressor - Test",
    style_reg,
    ell_test_part["text"],
    ell_test_part["style_target"]
)

Grammar test:

Grammar Regressor - Test
RMSE: 0.5007
MAE : 0.4041
R²  : 0.3522

Clarity test:

Clarity Regressor - Test
RMSE: 0.5007
MAE : 0.3969
R²  : 0.3496

Style test:

Style Regressor - Test
RMSE: 0.4358
MAE : 0.3460
R²  : 0.3783


In [41]:
%pip install -q transformers datasets accelerate sentencepiece scikit-learn

In [42]:
import pandas as pd

disc3_labels = ["structure", "evidence", "argument"]

disc3_df = pd.concat(
    [
        fp2021_samples[fp2021_samples["label"].isin(disc3_labels)],
        fpe_samples[fpe_samples["label"].isin(disc3_labels)]
    ],
    ignore_index=True
)

disc3_df["text"] = disc3_df["text"].map(clean_text)
disc3_df = disc3_df[disc3_df["text"].str.len() > 0].copy()

# Keep duplicate texts but aggregate source/weight instead of dropping blindly
disc3_df = (
    disc3_df
    .groupby(["text", "label"], as_index=False)
    .agg(
        weight=("weight", "mean"),
        source=("source", lambda s: "|".join(sorted(set(s))))
    )
)

print("3-class discourse dataset:", disc3_df.shape)
print(disc3_df["label"].value_counts())
display(disc3_df.sample(min(10, len(disc3_df)), random_state=SEED))

3-class discourse dataset: (138705, 4)
label
argument     70381
evidence     45607
structure    22717
Name: count, dtype: int64


,text,label,weight,source
104983,"We have some time for sight seeing, and on the...",evidence,1.0,fp2021_discourse
28348,He states that it has similar features to Eart...,evidence,1.0,fp2021_discourse|fpe_discourse
84415,The author does have some good evidence to bac...,argument,1.0,fp2021_discourse
45857,"In conclusion the ""face'' to me is still just ...",argument,1.0,fp2021_discourse|fpe_discourse
82499,The Electoral College is a process that should...,argument,1.0,fp2021_discourse|fpe_discourse
100564,This technology would help the students who wo...,evidence,1.0,fp2021_discourse
128203,people come to asmerica to be able to be free....,evidence,1.0,fp2021_discourse
57617,Making these cars cost losts of money,argument,1.0,fp2021_discourse|fpe_discourse
46742,"In conclusion, texting while driving is one of...",structure,1.0,fp2021_discourse
51367,It can also help teachers know why the kid is ...,argument,1.0,fp2021_discourse


In [43]:
from sklearn.model_selection import train_test_split

disc3_train, disc3_temp = train_test_split(
    disc3_df,
    test_size=0.20,
    stratify=disc3_df["label"],
    random_state=SEED
)

disc3_val, disc3_test = train_test_split(
    disc3_temp,
    test_size=0.50,
    stratify=disc3_temp["label"],
    random_state=SEED
)

print("Train:", disc3_train.shape)
print(disc3_train["label"].value_counts())

print("\nVal:", disc3_val.shape)
print(disc3_val["label"].value_counts())

print("\nTest:", disc3_test.shape)
print(disc3_test["label"].value_counts())

Train: (110964, 4)
label
argument     56305
evidence     36485
structure    18174
Name: count, dtype: int64

Val: (13870, 4)
label
argument     7038
evidence     4561
structure    2271
Name: count, dtype: int64

Test: (13871, 4)
label
argument     7038
evidence     4561
structure    2272
Name: count, dtype: int64


In [44]:
def rebalance_exact(df, target_per_class=12000, seed=42):
    parts = []
    for label, group in df.groupby("label"):
        if len(group) >= target_per_class:
            group = group.sample(target_per_class, random_state=seed)
        else:
            group = group.sample(target_per_class, replace=True, random_state=seed)
        parts.append(group)
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

disc3_train_bal = rebalance_exact(disc3_train, target_per_class=12000, seed=SEED)

print(disc3_train_bal.shape)
print(disc3_train_bal["label"].value_counts())

(36000, 4)
label
evidence     12000
structure    12000
argument     12000
Name: count, dtype: int64


In [45]:
disc3_label2id = {label: i for i, label in enumerate(disc3_labels)}
disc3_id2label = {i: label for label, i in disc3_label2id.items()}

disc3_label2id, disc3_id2label

({'structure': 0, 'evidence': 1, 'argument': 2},
 {0: 'structure', 1: 'evidence', 2: 'argument'})

In [46]:
from datasets import Dataset

train_hf = disc3_train_bal.copy()
val_hf = disc3_val.copy()
test_hf = disc3_test.copy()

train_hf["labels"] = train_hf["label"].map(disc3_label2id)
val_hf["labels"] = val_hf["label"].map(disc3_label2id)
test_hf["labels"] = test_hf["label"].map(disc3_label2id)

train_ds = Dataset.from_pandas(train_hf[["text", "labels", "weight"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_hf[["text", "labels", "weight"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_hf[["text", "labels", "weight"]], preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['text', 'labels', 'weight'],
    num_rows: 36000
})
Dataset({
    features: ['text', 'labels', 'weight'],
    num_rows: 13870
})
Dataset({
    features: ['text', 'labels', 'weight'],
    num_rows: 13871
})


In [47]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"   # or "distilroberta-base" if memory is tight

disc3_labels = ["structure", "evidence", "argument"]
disc3_label2id = {label: i for i, label in enumerate(disc3_labels)}
disc3_id2label = {i: label for label, i in disc3_label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_hf = disc3_train_bal.copy()
val_hf = disc3_val.copy()
test_hf = disc3_test.copy()

train_hf["labels"] = train_hf["label"].map(disc3_label2id)
val_hf["labels"] = val_hf["label"].map(disc3_label2id)
test_hf["labels"] = test_hf["label"].map(disc3_label2id)

train_ds = Dataset.from_pandas(train_hf[["text", "labels", "weight"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_hf[["text", "labels", "weight"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_hf[["text", "labels", "weight"]], preserve_index=False)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

print(train_ds.column_names)
print(val_ds.column_names)
print(test_ds.column_names)

Map:   0%|          | 0/36000 [00:00<?, ? examples/s]

Map:   0%|          | 0/13870 [00:00<?, ? examples/s]

Map:   0%|          | 0/13871 [00:00<?, ? examples/s]

['labels', 'weight', 'input_ids', 'attention_mask']
['labels', 'weight', 'input_ids', 'attention_mask']
['labels', 'weight', 'input_ids', 'attention_mask']


In [49]:
import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.cuda.is_available(): False
torch.cuda.device_count(): 0


In [50]:
disc3_labels = ["structure", "evidence", "argument"]

disc3_df = combined_df[combined_df["label"].isin(disc3_labels)].copy()

print("3-class discourse dataset:", disc3_df.shape)
print(disc3_df["label"].value_counts())

3-class discourse dataset: (138705, 4)
label
argument     70381
evidence     45607
structure    22717
Name: count, dtype: int64


In [51]:
from sklearn.model_selection import train_test_split

disc3_train, disc3_temp = train_test_split(
    disc3_df,
    test_size=0.20,
    stratify=disc3_df["label"],
    random_state=SEED
)

disc3_val, disc3_test = train_test_split(
    disc3_temp,
    test_size=0.50,
    stratify=disc3_temp["label"],
    random_state=SEED
)

print("Train:")
print(disc3_train["label"].value_counts())
print("\nVal:")
print(disc3_val["label"].value_counts())
print("\nTest:")
print(disc3_test["label"].value_counts())

Train:
label
argument     56305
evidence     36485
structure    18174
Name: count, dtype: int64

Val:
label
argument     7038
evidence     4561
structure    2271
Name: count, dtype: int64

Test:
label
argument     7038
evidence     4561
structure    2272
Name: count, dtype: int64


In [52]:
def rebalance_exact(df, target_per_class=12000, seed=42):
    parts = []
    for label, group in df.groupby("label"):
        if len(group) >= target_per_class:
            group = group.sample(target_per_class, random_state=seed)
        else:
            group = group.sample(target_per_class, replace=True, random_state=seed)
        parts.append(group)
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

disc3_train_bal = rebalance_exact(disc3_train, target_per_class=12000, seed=SEED)

print(disc3_train_bal.shape)
print(disc3_train_bal["label"].value_counts())

(36000, 4)
label
evidence     12000
structure    12000
argument     12000
Name: count, dtype: int64


In [53]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

disc3_feature_block = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="word",
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=140000
    )),
    ("char_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="char_wb",
        ngram_range=(3, 6),
        min_df=2,
        sublinear_tf=True,
        max_features=100000
    ))
])

def eval_cls(name, model, X, y, labels):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    macro = f1_score(y, preds, average="macro")
    weighted = f1_score(y, preds, average="weighted")

    print(f"\n{'='*80}")
    print(name)
    print(f"{'='*80}")
    print(f"Accuracy   : {acc:.4f}")
    print(f"Macro F1   : {macro:.4f}")
    print(f"Weighted F1: {weighted:.4f}")
    print("\nClassification report:\n")
    print(classification_report(y, preds, labels=labels, digits=4))
    return preds, acc, macro, weighted

In [54]:
disc3_svm = Pipeline([
    ("features", disc3_feature_block),
    ("clf", LinearSVC(
        C=1.5,
        class_weight="balanced",
        random_state=SEED
    ))
])

disc3_svm.fit(
    disc3_train_bal["text"],
    disc3_train_bal["label"],
    clf__sample_weight=disc3_train_bal["weight"].values
)

svm_preds, svm_acc, svm_macro, svm_weighted = eval_cls(
    "3-Class Discourse + LinearSVC",
    disc3_svm,
    disc3_val["text"],
    disc3_val["label"],
    disc3_labels
)


3-Class Discourse + LinearSVC
Accuracy   : 0.7518
Macro F1   : 0.7242
Weighted F1: 0.7570

Classification report:

              precision    recall  f1-score   support

   structure     0.5441    0.7252    0.6217      2271
    evidence     0.7386    0.7360    0.7373      4561
    argument     0.8612    0.7707    0.8134      7038

    accuracy                         0.7518     13870
   macro avg     0.7146    0.7440    0.7242     13870
weighted avg     0.7690    0.7518    0.7570     13870



In [55]:
disc3_logreg = Pipeline([
    ("features", disc3_feature_block),
    ("clf", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=SEED
    ))
])

disc3_logreg.fit(
    disc3_train_bal["text"],
    disc3_train_bal["label"],
    clf__sample_weight=disc3_train_bal["weight"].values
)

log_preds, log_acc, log_macro, log_weighted = eval_cls(
    "3-Class Discourse + LogisticRegression",
    disc3_logreg,
    disc3_val["text"],
    disc3_val["label"],
    disc3_labels
)


3-Class Discourse + LogisticRegression
Accuracy   : 0.7720
Macro F1   : 0.7427
Weighted F1: 0.7764

Classification report:

              precision    recall  f1-score   support

   structure     0.5671    0.7252    0.6365      2271
    evidence     0.7597    0.7564    0.7581      4561
    argument     0.8732    0.7971    0.8334      7038

    accuracy                         0.7720     13870
   macro avg     0.7333    0.7596    0.7427     13870
weighted avg     0.7858    0.7720    0.7764     13870



In [56]:
best_model = disc3_svm if svm_weighted >= log_weighted else disc3_logreg
best_name = "LinearSVC" if svm_weighted >= log_weighted else "LogisticRegression"

print("Best model:", best_name)

Best model: LogisticRegression


In [57]:
test_preds = best_model.predict(disc3_test["text"])

print(f"\nBest model: {best_name}")
print(f"Test Accuracy   : {accuracy_score(disc3_test['label'], test_preds):.4f}")
print(f"Test Macro F1   : {f1_score(disc3_test['label'], test_preds, average='macro'):.4f}")
print(f"Test Weighted F1: {f1_score(disc3_test['label'], test_preds, average='weighted'):.4f}")
print("\nTest report:\n")
print(classification_report(disc3_test["label"], test_preds, labels=disc3_labels, digits=4))


Best model: LogisticRegression
Test Accuracy   : 0.7747
Test Macro F1   : 0.7459
Test Weighted F1: 0.7793

Test report:

              precision    recall  f1-score   support

   structure     0.5657    0.7390    0.6408      2272
    evidence     0.7704    0.7520    0.7611      4561
    argument     0.8738    0.8009    0.8358      7038

    accuracy                         0.7747     13871
   macro avg     0.7367    0.7640    0.7459     13871
weighted avg     0.7894    0.7747    0.7793     13871



In [58]:
import pandas as pd

def rebalance_discourse_focus(df, seed=42):
    targets = {
        "structure": 18000,
        "evidence": 12000,
        "argument": 10000,
    }

    parts = []
    for label, group in df.groupby("label"):
        target = targets[label]
        if len(group) >= target:
            group = group.sample(target, random_state=seed)
        else:
            group = group.sample(target, replace=True, random_state=seed)
        parts.append(group)

    out = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

disc3_train_focus = rebalance_discourse_focus(disc3_train, seed=SEED)

print(disc3_train_focus.shape)
print(disc3_train_focus["label"].value_counts())

(40000, 4)
label
structure    18000
evidence     12000
argument     10000
Name: count, dtype: int64


In [59]:
class_weight_focus = {
    "structure": 1.8,
    "evidence": 1.0,
    "argument": 0.9,
}